# 🤝 Handoffs with `langgraph-swarm`

## Learning Objectives
In this notebook, you will learn:
1. **Handoff vs supervisor** - In a handoff, the agent that holds the conversation passes it on, and the receiver keeps it
2. **Handoff tools** - Give each prebuilt `create_agent` agent `transfer_to_<agent>` tools from `create_handoff_tool`
3. **The swarm** - Join the agents with `create_swarm`, which routes every new message to the **active agent**
4. **Memory of who is active** - Compile with a checkpointer so a follow-up message on the same thread goes straight to the specialist
5. **Loop guard** - Stop agents that keep handing off to each other with `recursion_limit`, proved without an LLM

## Where this fits
This is pattern 5 of [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md). In `02_supervisor_agents_as_tools.ipynb`, control always returned to the supervisor. Here no agent is in charge: an agent decides to **transfer** the conversation, and the specialist that receives it becomes the active agent. Module 07's `01_agent_handoffs.ipynb` builds the same mechanism by hand; this notebook uses the library.

**Flow:** `User → triage agent → handoff → active specialist → conversation continues`

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env` and sets `RECURSION_LIMIT`, the step cap every run passes (Part 4).

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and imports
# ============================================================================
import itertools
import uuid
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph_swarm import create_handoff_tool, create_swarm

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every run passes this cap. A handoff system has no coordinator that can say
# "stop", so this cap is what ends an endless handoff loop (Part 4).
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔀 Part 1: Handoff vs Supervisor

| | Supervisor (notebook 02) | Handoff (this notebook) |
|---|---|---|
| Who picks the next agent | always the supervisor | the agent that currently holds the conversation |
| After a specialist finishes | control returns to the supervisor | the specialist keeps the conversation |
| The user's next message goes to | the supervisor again | the active agent |
| What a specialist sees | only the request the supervisor wrote | the whole conversation |

`langgraph-swarm` implements handoffs with two functions:
- **`create_handoff_tool(agent_name=...)`** builds a tool named `transfer_to_<agent_name>`. When an agent calls it, the tool returns `Command(goto=agent_name, graph=Command.PARENT, ...)`: "parent graph, run that agent next". It also writes `active_agent = agent_name` into the state.
- **`create_swarm(agents, default_active_agent=...)`** builds the parent graph. Its `START` router sends each new message to `active_agent`, or to the default agent on a new thread.

---
## 🛠️ Part 2: Three Agents and Their Tools

A customer service desk for a cloud storage app:

| Agent | Own tools | Can hand off to |
|---|---|---|
| `triage_agent` | none | `billing_agent`, `tech_agent` |
| `billing_agent` | `lookup_invoice`, `file_refund` | `tech_agent`, `triage_agent` |
| `tech_agent` | `check_service_status` | `billing_agent`, `triage_agent` |

Each agent is a prebuilt `create_agent` agent. Its `name` must match the `agent_name` used in the handoff tools, because the swarm uses that name as the node name.

In [ ]:
# ============================================================================
# DOMAIN TOOLS: in-memory invoices and service status
# ============================================================================
INVOICES = {
    "INV-1001": "₹499 charged on 2026-10-01 for the Pro plan (monthly).",
    "INV-1002": "₹499 charged on 2026-10-01 for the Pro plan (monthly). Flagged: duplicate of INV-1001.",
}
SERVICE_STATUS = {
    "sync": "Degraded: file sync delayed by up to 30 minutes since 09:00 IST. Engineers are working on it.",
    "login": "Operational.",
    "uploads": "Operational.",
}


@tool
def lookup_invoice(invoice_id: str) -> str:
    """Look up an invoice by its ID, for example INV-1001."""
    return INVOICES.get(invoice_id.strip().upper(), f"No invoice found with ID {invoice_id}.")


@tool
def file_refund(invoice_id: str, reason: str) -> str:
    """File a refund request for an invoice. Returns the refund reference."""
    return f"Refund request REF-{invoice_id.strip().upper()} filed ({reason}). Money returns in 5-7 working days."


@tool
def check_service_status(service: str) -> str:
    """Check the live status of a service: sync, login or uploads."""
    return SERVICE_STATUS.get(service.strip().lower(), f"Unknown service {service}. Known: {', '.join(SERVICE_STATUS)}.")


print(lookup_invoice.invoke({"invoice_id": "INV-1002"}))
print(check_service_status.invoke({"service": "sync"}))

In [ ]:
# ============================================================================
# HANDOFF TOOLS AND AGENTS
# ============================================================================
to_billing = create_handoff_tool(
    agent_name="billing_agent",
    description="Transfer the customer to billing for payments, charges, invoices and refunds.",
)
to_tech = create_handoff_tool(
    agent_name="tech_agent",
    description="Transfer the customer to technical support for sync, login, upload and app problems.",
)
to_triage = create_handoff_tool(
    agent_name="triage_agent",
    description="Transfer the customer back to triage for anything outside your area.",
)

triage_agent = create_agent(
    llm,
    tools=[to_billing, to_tech],
    system_prompt=(
        "You are the triage agent of a cloud storage company. You do not solve problems yourself. "
        "Transfer billing questions to billing and technical problems to technical support. "
        "Only answer a question yourself if it fits neither."
    ),
    name="triage_agent",
)
billing_agent = create_agent(
    llm,
    tools=[lookup_invoice, file_refund, to_tech, to_triage],
    system_prompt=(
        "You are the billing specialist. Use your tools to look up invoices and file refunds. "
        "Read the conversation so far and never ask for details the customer already gave. "
        "If the customer raises a technical problem, transfer them to technical support. "
        "Reply in at most 3 sentences."
    ),
    name="billing_agent",
)
tech_agent = create_agent(
    llm,
    tools=[check_service_status, to_billing, to_triage],
    system_prompt=(
        "You are the technical support specialist. Check the service status before answering. "
        "Read the conversation so far and never ask for details the customer already gave. "
        "If the customer raises a billing question, transfer them to billing. "
        "Reply in at most 3 sentences."
    ),
    name="tech_agent",
)

print("✅ Agents ready:", [t.name for t in [to_billing, to_tech, to_triage]])

---
## 🐝 Part 3: Build the Swarm and Follow One Conversation

`create_swarm` returns an uncompiled `StateGraph`. Compiling it with `InMemorySaver()` saves the state, the messages **and** `active_agent`, per `thread_id`. Without a checkpointer, every message would start again at `triage_agent`.

`run_turn` sends one message and reads only the messages this turn added:
- **agents this turn** - the `name` that `create_agent` stamps on each `AIMessage`, in order
- **🔀 lines** - the `ToolMessage` each handoff tool wrote
- **active_agent** - who holds the conversation after the turn

In [ ]:
# ============================================================================
# BUILD THE SWARM: triage is the default active agent
# ============================================================================
swarm_app = create_swarm(
    [triage_agent, billing_agent, tech_agent],
    default_active_agent="triage_agent",
).compile(checkpointer=InMemorySaver())


def run_turn(app, thread_id: str, text: str):
    """Send one message on a thread; print who acted; return (agents this turn, active_agent)."""
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": RECURSION_LIMIT}
    result = app.invoke({"messages": [HumanMessage(content=text)]}, config)

    last_human = max(i for i, m in enumerate(result["messages"]) if isinstance(m, HumanMessage))
    agents = []  # agent names in the order they acted, consecutive repeats collapsed
    print("=" * 80)
    print(f"👤 {text}")
    for message in result["messages"][last_human + 1 :]:
        if isinstance(message, AIMessage) and (not agents or agents[-1] != message.name):
            agents.append(message.name)
        if isinstance(message, ToolMessage) and (message.name or "").startswith("transfer_to_"):
            print(f"🔀 {message.content}")
    print(f"🤖 {result['messages'][-1].name}: {result['messages'][-1].content}")
    print(f"agents this turn: {' → '.join(agents)} | active_agent: {result['active_agent']}")
    return agents, result["active_agent"]


print(swarm_app.get_graph().draw_mermaid())

### 3.1 ▶️ One Thread, Three Turns

1. A double charge → triage hands off to **billing**, which looks up the invoice
2. A follow-up → starts at **billing**; triage does not run, because billing is the active agent
3. A sync problem → billing hands off **sideways** to **tech**

In [ ]:
# ============================================================================
# RUN: ONE THREAD, THREE TURNS
# ============================================================================
thread_id = str(uuid.uuid4())
SCRIPT = [  # (customer message, expected agents this turn, expected active agent after it)
    ("Hi, I think I was charged twice this month. The second invoice is INV-1002.",
     ["triage_agent", "billing_agent"], "billing_agent"),
    ("Yes please, refund the duplicate charge.",
     ["billing_agent"], "billing_agent"),
    ("Thanks. Separately, my files have not synced since this morning.",
     ["billing_agent", "tech_agent"], "tech_agent"),
]

passed = 0
for text, expected_agents, expected_active in SCRIPT:
    agents, active = run_turn(swarm_app, thread_id, text)
    ok = agents == expected_agents and active == expected_active
    passed += ok
    print(f"{'✅' if ok else '❌'} expected {' → '.join(expected_agents)}, active {expected_active}")

print(f"\n{passed}/{len(SCRIPT)} turns took the expected route")

Turn 2 is the point of the checkpointer: the first agent to act is `billing_agent`, because the swarm's `START` router read the saved `active_agent`. Billing also did not ask for the invoice ID again: a handoff forwards the **whole conversation**, unlike the supervisor's tool call in notebook 02, which forwards only the request it writes. Turn 3 shows control moving sideways between specialists, with no coordinator involved.

On a new `thread_id` nothing is saved, so the next message would start at `triage_agent` again.

---
## 🛡️ Part 4: Guard Against Handoff Loops

No node in a swarm can say "stop". If two agents each decide the other should handle a request, they can transfer it back and forth forever. Each agent run is one super-step of the swarm graph, so `recursion_limit` in the run config caps the loop and raises `GraphRecursionError`.

To see the guard without paying for a loop, the agents below run on a **scripted stand-in model**. `ScriptedModel` is LangChain's `GenericFakeChatModel`, which replays messages from an iterator, with `bind_tools` returning the model itself: `create_agent` binds the tools, and the scripted replies already contain their tool calls. The script makes every agent hand off immediately: triage → billing → tech → triage → … The same `create_handoff_tool` and `create_swarm` calls build the graph; only the model differs.

In [ ]:
# ============================================================================
# LOOP GUARD: endless handoffs stopped by recursion_limit (no LLM)
# ============================================================================
class ScriptedModel(GenericFakeChatModel):
    """A stand-in chat model: replays scripted replies, makes no API call."""

    def bind_tools(self, tools, **kwargs):
        return self  # the scripted replies already contain their tool calls


handoffs = []  # one entry per scripted handoff


def endless_handoffs():
    """Scripted replies: transfer to billing, tech, triage, billing, ... forever."""
    for target in itertools.cycle(["billing_agent", "tech_agent", "triage_agent"]):
        handoffs.append(target)
        yield AIMessage(content="", tool_calls=[{"name": f"transfer_to_{target}", "args": {}, "id": f"call_{len(handoffs)}"}])


scripted = ScriptedModel(messages=endless_handoffs())  # one shared iterator serves every agent in call order
looping_app = create_swarm(
    [
        create_agent(scripted, tools=[to_billing, to_tech], name="triage_agent"),
        create_agent(scripted, tools=[to_tech, to_triage], name="billing_agent"),
        create_agent(scripted, tools=[to_billing, to_triage], name="tech_agent"),
    ],
    default_active_agent="triage_agent",
).compile()

LOOP_LIMIT = 10  # a small cap for this demo; the real runs above use RECURSION_LIMIT
try:
    looping_app.invoke({"messages": [HumanMessage(content="Help!")]}, {"recursion_limit": LOOP_LIMIT})
    print("❌ the loop ended on its own")
except GraphRecursionError as error:
    print(f"✅ GraphRecursionError after {len(handoffs)} handoffs: {str(error).splitlines()[0]}")

In a real system, keep the cap small enough to fail fast and large enough for the longest legitimate route. Clear handoff descriptions and prompts that say *when not to transfer* make loops rare; the cap makes them harmless.

---
## 📝 Summary

In this notebook, we learned:

### 1. Handoff vs supervisor
- A supervisor takes control back after every specialist; a handoff passes control on, and the receiver becomes the active agent.

### 2. Handoff tools
- `create_handoff_tool(agent_name=...)` builds `transfer_to_<agent>`, which returns `Command(goto=..., graph=Command.PARENT)` and sets `active_agent`.

### 3. The swarm
- `create_swarm([...], default_active_agent=...)` joins prebuilt `create_agent` agents and routes each new message to the active agent. Agent `name`s must match the handoff targets.

### 4. Memory of who is active
- With a checkpointer, `active_agent` outlives the turn, so a follow-up skips triage. A handoff forwards the whole conversation, so specialists need not ask again.

### 5. Loop guard
- `recursion_limit` turns an endless handoff loop into a `GraphRecursionError`, proved with a scripted model and no LLM.

### Next Steps
- Revisit [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md) and pick the pattern that fits your next task: a fixed workflow first, then a supervisor, parallel workers, a refinement loop or handoffs only when the task needs them